# Additionnel · RQ-014 jouée sur les vraies données

**Règle** (`dq.regle`) : *RQ-014, adresse de sinistre : format valide, rapprochée du référentiel national (BAN), enregistrement bloqué si échec.* Bloquante, porte S, seuil 100 %.

| Étape | Ce qu'on fait | Écrit en base ? |
|---|---|---|
| **1** | Tester `silver.sinistre` avec RQ-014 sur l'**API de la BAN** (état initial) | non (lecture) |
| **2** | **Nettoyer** les adresses pour coller au référentiel BAN (normalisation, regex) | non (mémoire) |
| **3** | **Retester** sur l'API, puis **quarantaine** des lignes qui ne correspondent toujours pas | oui : `silver.sinistre_r014`, `silver.sinistre_r014_trace`, `dq.rejets` |
| **4** | Repasser en **gold**, mesurer, émettre le **certificat**, publier la **nouvelle table cible** | oui : `gold.sinistre_certifie_r014`, `dq.dq_history`, `dq.certificat_qualite`, `crm_cible.sinistre_r014` |

**Lot de base intact.** Tout est écrit sous un **nouveau lot** (`<lot de base>-R14`) et dans des **tables nouvelles** suffixées `_r014`. Rien n'est modifié dans `silver.sinistre`, `gold.*`, `crm_cible.sinistre` ni dans le certificat du lot de base. S2 et S4 ne sont pas relancés. La section *Annexe* retire tout.

**Contraintes** : aucune bibliothèque à installer (`urllib`, `json`, `difflib`, `re`… + `pandas` et `aq.py` déjà présents).
**Prérequis** : S4 et S5 exécutés (silver, gold, certificat du lot de base) ; accès Internet vers `data.geopf.fr`.
**Durée** : environ 10 à 15 minutes (≈ 14 000 appels à l'API la première fois ; relancer le notebook réutilise le cache `reports/ban_cache.json`).

## 0. Paramètres et outils

In [38]:
# -*- coding: utf-8 -*-
import sys, os, re, json, time, math, unicodedata, threading, difflib, hashlib
import urllib.request, urllib.parse, urllib.error
from concurrent.futures import ThreadPoolExecutor
import numpy as np
sys.path.append("../pipelines")
from pipelines.aq import *                       # lire, executer, ecrire, eng, text, BATCH, RACINE, pd (déjà installés : aucune nouvelle bibliothèque)

# ---------- paramètres ----------
URL_BAN      = "https://data.geopf.fr/geocodage/search"   # géocodage de l'IGN (successeur de api-adresse.data.gouv.fr)
SEUIL_VOIE   = 0.85     # similarité minimale (0 à 1) entre la voie saisie et la voie renvoyée par la BAN
NUMERO_OBLIGATOIRE = True   # une adresse sans numéro est non conforme (comme en S4, US-26)
NETTOYAGE_FLOU     = True   # True : corrige aussi les fautes de frappe (règles M et V). False : seulement les règles sûres (F, C, A, T)
CRITERE      = "API_OU_LOCAL"    # critère de quarantaine : "API"  = voie reconnue par la BAN
                        #                          "API_OU_LOCAL" = reconnue par la BAN OU présente (numéro compris) dans l'extrait local silver.ref_ban
ECH_NUMERO   = 300      # sinistres dont on contrôle aussi le numéro (diagnostic, 0 : sauter)
ECH_REFERENTIEL = 100   # adresses du référentiel local passées à la BAN (diagnostic, 0 : sauter)
MAX_APPELS   = 25000    # garde-fou : une étape s'arrête avant d'appeler plus que cela
DEBIT_MAX    = 30       # requêtes par seconde (la BAN en accepte 50 par adresse IP)
THREADS      = 12
SEED         = 42
ECRIRE       = True     # False : rien n'est écrit en base (les étapes 3 et 4 affichent ce qu'elles écriraient)

LOT  = f"{BATCH}-R14"   # nouveau lot de travail ; le lot de base reste BATCH
CERT = f"CERT-{LOT}"
assert LOT != BATCH and len(LOT) <= 20 and len(CERT) <= 30, "identifiant de lot trop long pour dq.dq_history / dq.certificat_qualite"
assert CRITERE in ("API", "API_OU_LOCAL")

CACHE = f"{RACINE}/reports/ban_cache.json"
os.makedirs(os.path.dirname(CACHE), exist_ok=True)
def charger_cache():
    try:
        with open(CACHE, encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):
        return {}
cache = charger_cache()
def sauver_cache():
    """Écriture atomique : le fichier n'est jamais à moitié écrit."""
    with open(CACHE + ".tmp", "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)
    os.replace(CACHE + ".tmp", CACHE)
print(f"Lot de base {BATCH} (intact) · lot de travail {LOT} · critère {CRITERE} · écriture en base : {ECRIRE}")
print(f"API {URL_BAN} · cache : {len(cache)} réponses ({CACHE})")

Lot de base L-2026-10-01 (intact) · lot de travail L-2026-10-01-R14 · critère API_OU_LOCAL · écriture en base : True
API https://data.geopf.fr/geocodage/search · cache : 14235 réponses (/home/jovyan/work/reports/ban_cache.json)


### Outils : appel de l'API et règle de décision

La BAN renvoie, pour une requête, le meilleur résultat avec son **type** : `housenumber` (le numéro existe), `street` (la voie existe, pas le numéro), `locality`, `municipality`.
Une adresse est **reconnue au niveau de la voie** si le premier résultat est une voie ou un numéro, avec le **même code postal**, la **même commune** et une **voie ressemblante** (similarité ≥ `SEUIL_VOIE`, abréviations développées : R = RUE…).
C'est le niveau retenu pour RQ-014 : l'étape 1 montre pourquoi le niveau « numéro » n'est pas exploitable avec les données du TD.

In [39]:
ABREV = {"R": "RUE", "AV": "AVENUE", "AVE": "AVENUE", "BD": "BOULEVARD", "BLD": "BOULEVARD", "CHE": "CHEMIN", "IMP": "IMPASSE",
         "PL": "PLACE", "ALL": "ALLEE", "RTE": "ROUTE", "SQ": "SQUARE", "CRS": "COURS", "ST": "SAINT", "STE": "SAINTE"}

def norm(s):
    """Comparaison : majuscules, sans accents, ponctuation et tirets remplacés par des espaces, abréviations courantes développées."""
    s = unicodedata.normalize("NFKD", "" if s is None or (isinstance(s, float) and math.isnan(s)) else str(s)).encode("ascii", "ignore").decode().upper()
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    return " ".join(ABREV.get(m, m) for m in s.split())

# ---------- appel de l'API (urllib : bibliothèque standard) ----------
_verrou, _dernier = threading.Lock(), [0.0]
def _creneau():
    """Répartit les appels dans le temps : au plus DEBIT_MAX requêtes par seconde, tous threads confondus."""
    with _verrou:
        t = time.monotonic()
        _dernier[0] = max(t, _dernier[0] + 1.0 / DEBIT_MAX)
        attente = _dernier[0] - t
    if attente > 0:
        time.sleep(attente)

def reduire(rep):
    """On ne garde du JSON que ce qui sert à la décision : le cache reste petit et relisible."""
    cles = ("type", "housenumber", "street", "name", "postcode", "city", "score")
    return {"features": [{"properties": {k: f["properties"].get(k) for k in cles}} for f in (rep.get("features") or [])[:1]]}

def appeler(q, cp):
    """Premier résultat de la BAN pour la requête q, filtré sur le code postal cp. Réponse mise en cache (clé : cp|q)."""
    cle = f"{cp}|{q}"
    if cle in cache:
        return cache[cle]
    url = URL_BAN + "?" + urllib.parse.urlencode({"q": q, "postcode": cp, "limit": 1})
    derniere = None
    for essai in range(4):
        _creneau()
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "TD-AssurConnect-RQ014/2.0"})
            with urllib.request.urlopen(req, timeout=15) as r:
                cache[cle] = reduire(json.load(r))
            return cache[cle]
        except urllib.error.HTTPError as e:
            derniere = e
            if e.code == 400:                                  # requête refusée (trop courte, caractères) : pas de résultat
                cache[cle] = {"features": []}
                return cache[cle]
            time.sleep(2 ** essai)                             # 429, 5xx : on patiente puis on réessaie
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            derniere = e
            time.sleep(2 ** essai)
    raise RuntimeError(f"API BAN injoignable après 4 essais ({derniere}). Relancez la cellule : les réponses déjà reçues sont dans le cache.")

# ---------- règle de décision ----------
def evaluer(rep, cp, ville, voie, numero=None):
    """Compare le premier résultat de la BAN à l'adresse saisie."""
    f = (rep or {}).get("features") or []
    if not f:
        return {"cause": "aucun résultat (voie ou couple code postal / commune inconnu)", "voie_ok": False, "numero_ok": False, "voie_ban": None, "type_ban": None}
    p = f[0]["properties"]
    voie_ban = p.get("street") or p.get("name") or ""
    sim = difflib.SequenceMatcher(None, norm(voie), norm(voie_ban)).ratio()
    cp_ok = str(p.get("postcode")) == str(cp)
    ville_ok = norm(p.get("city")) == norm(ville)
    voie_ok = p.get("type") in ("street", "housenumber") and cp_ok and ville_ok and sim >= SEUIL_VOIE
    numero_ok = bool(voie_ok and numero is not None and p.get("type") == "housenumber" and str(p.get("housenumber")) == str(numero))
    cause = "reconnue" if voie_ok else ("code postal différent" if not cp_ok else "commune différente" if not ville_ok else "voie différente")
    return {"cause": cause, "voie_ok": bool(voie_ok), "numero_ok": numero_ok, "voie_ban": voie_ban, "type_ban": p.get("type")}

def controler_voie(r):
    cp, ville, voie = r
    return evaluer(appeler(f"{voie} {ville}", cp), cp, ville, voie)

def controler_numero(r):
    cp, ville, voie, numero = r
    return evaluer(appeler(f"{numero} {voie} {ville}", cp), cp, ville, voie, numero)

def a_appeler(couples, avec_numero=False):
    """Nombre d'appels réellement nécessaires (ceux qui ne sont pas déjà dans le cache)."""
    if avec_numero:
        return sum(f"{cp}|{n} {v} {ville}" not in cache for cp, ville, v, n in couples)
    return sum(f"{cp}|{v} {ville}" not in cache for cp, ville, v in couples)

def interroger(couples, fonction, titre, pas=500):
    """Interroge l'API pour une liste de tuples, par lots : on voit avancer et le cache est sauvé à chaque lot."""
    t0, sortie = time.time(), []
    try:
        with ThreadPoolExecutor(THREADS) as ex:
            for i in range(0, len(couples), pas):
                sortie += list(ex.map(fonction, couples[i:i + pas]))
                if (i // pas) % 4 == 3 or i + pas >= len(couples):
                    print(f"  {titre} : {min(i + pas, len(couples)):>6} / {len(couples)} · {time.time() - t0:.0f} s")
                sauver_cache()
    finally:
        sauver_cache()
    return sortie

def wilson(k, n, z=1.96):
    """Intervalle de confiance à 95 % d'une proportion k/n (en %)."""
    if n == 0:
        return (None, None)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (round(100 * (c - h), 1), round(100 * (c + h), 1))

# ---------- auto-test de la règle de décision (sans réseau) ----------
def _f(type_, street, cp, city, hn=None):
    return {"features": [{"properties": {"type": type_, "street": street, "postcode": cp, "city": city, "housenumber": hn}}]}
assert evaluer(_f("housenumber", "Rue de la Paix", "75002", "Paris", "12"), "75002", "PARIS", "R DE LA PAIX", "12")["numero_ok"]
assert evaluer(_f("street", "Rue de la Paix", "75002", "Paris"), "75002", "PARIS", "RUE DE LA PAIX", "9999")["voie_ok"]
assert not evaluer(_f("street", "Rue de la Paix", "75002", "Paris"), "75002", "PARIS", "RUE DE LA PAIX", "9999")["numero_ok"]
assert evaluer(_f("street", "Rue de la Paix", "75002", "Paris"), "75003", "PARIS", "RUE DE LA PAIX")["cause"] == "code postal différent"
assert evaluer(_f("street", "Rue Molière", "01000", "Bourg-en-Bresse"), "01000", "LYON", "RUE MOLIERE")["cause"] == "commune différente"
assert evaluer(_f("street", "Rue Molière", "01000", "Bourg-en-Bresse"), "01000", "BOURG EN BRESSE", "RUE VICTOR HUGO")["cause"] == "voie différente"
assert evaluer({"features": []}, "01000", "X", "Y")["voie_ok"] is False
assert norm("Saint-Étienne") == "SAINT ETIENNE" and norm("ST ETIENNE") == "SAINT ETIENNE"
print("Règle de décision : 8 cas vérifiés.")

Règle de décision : 8 cas vérifiés.


---
# Étape 1 · Tester `silver.sinistre` avec RQ-014 sur l'API de la BAN

On mesure l'**état initial**, avant tout nettoyage. Deux volets de la règle :
- **format valide** : code postal à 5 chiffres, commune et voie renseignées, numéro présent (aucun appel nécessaire) ;
- **rapprochement BAN** : une requête par **voie distincte** (code postal + commune + voie), partagée par tous les sinistres de cette voie.

In [40]:
try:
    t0 = time.time()
    essai = appeler("8 bd du port Amiens", "80000")
    p = essai["features"][0]["properties"] if essai["features"] else {}
    print(f"API joignable en {time.time() - t0:.2f} s · {p.get('type')} · {p.get('housenumber')} {p.get('street')} {p.get('postcode')} {p.get('city')}")
except RuntimeError as e:
    print("Pas d'accès à l'API depuis ce conteneur :", e)
    print(f"Le notebook continue avec le cache ({len(cache)} réponses) ; les adresses qui n'y sont pas ne pourront pas être contrôlées.")
sauver_cache()

API joignable en 0.00 s · housenumber · 8 Boulevard du Port 80000 Amiens


### 1.1 Les adresses de sinistre en silver : format, puis voies à interroger

In [41]:
sin = lire("SELECT silver_row_id, sinistre_id, canal_saisie, cp, ville_normalisee AS ville, adresse_normalisee FROM silver.sinistre")
sin["adresse_normalisee"] = sin.adresse_normalisee.fillna("").astype(str).str.strip()
sin["adresse_avant"] = sin.adresse_normalisee
sin["numero"] = sin.adresse_normalisee.str.extract(r"^(\d+)")[0]
sin["voie"] = sin.adresse_normalisee.str.replace(r"^\d+\s+", "", regex=True).str.strip()
sin["cp"] = sin.cp.fillna("").astype(str).str.strip()
sin["ville"] = sin.ville.fillna("").astype(str).str.strip()

sin["format_ok"] = sin.cp.str.fullmatch(r"[0-9]{5}") & (sin.voie != "") & (sin.ville != "") & (sin.numero.notna() if NUMERO_OBLIGATOIRE else True)
sin["cause_avant"] = None
sin.loc[~sin.cp.str.fullmatch(r"[0-9]{5}"), "cause_avant"] = "format : code postal sans 5 chiffres"
sin.loc[sin.cause_avant.isna() & (sin.adresse_normalisee == ""), "cause_avant"] = "format : adresse vide"
sin.loc[sin.cause_avant.isna() & NUMERO_OBLIGATOIRE & sin.numero.isna(), "cause_avant"] = "format : pas de numéro"
sin.loc[sin.cause_avant.isna() & ((sin.voie == "") | (sin.ville == "")), "cause_avant"] = "format : voie ou commune vide"

rues = sin.loc[sin.format_ok, ["cp", "ville", "voie"]].drop_duplicates().reset_index(drop=True)
couples = list(zip(rues.cp, rues.ville, rues.voie))
n_appels = a_appeler(couples)
print(f"{len(sin):,} sinistres en silver".replace(",", " "))
print(f"  format non conforme (sans appel) : {int((~sin.format_ok).sum()):,}".replace(",", " "))
print(sin.loc[~sin.format_ok, "cause_avant"].value_counts().to_string())
print(f"{len(rues):,} voies distinctes à contrôler, dont {n_appels:,} pas encore en cache (≈ {n_appels / min(DEBIT_MAX, THREADS / 0.45) / 60:.1f} min)".replace(",", " "))
assert n_appels <= MAX_APPELS, f"{n_appels} appels prévus : augmentez MAX_APPELS si c'est voulu."

100 000 sinistres en silver
  format non conforme (sans appel) : 3 947
cause_avant
format : pas de numéro           1982
format : voie ou commune vide    1965
13 831 voies distinctes à contrôler  dont 0 pas encore en cache (≈ 0.0 min)


### 1.2 Contrôle des voies auprès de la BAN

In [42]:
t0 = time.time()
resultats = interroger(couples, controler_voie, "voies")
taille = f"fichier {os.path.getsize(CACHE) / 1e6:.1f} Mo" if os.path.exists(CACHE) else "⚠ fichier de cache introuvable sur le disque (les réponses restent en mémoire)"
print(f"Terminé en {time.time() - t0:.0f} s · {len(cache)} réponses en cache · {taille}")

res = pd.concat([rues, pd.DataFrame(resultats)], axis=1).rename(columns={"cause": "cause_ban"})
sin = sin.merge(res, on=["cp", "ville", "voie"], how="left")
sin["conforme_avant"] = sin.format_ok & sin.voie_ok.eq(True)           # une voie jamais interrogée (format non conforme) reste non conforme
sin.loc[sin.format_ok & ~sin.conforme_avant, "cause_avant"] = "BAN : " + sin.cause_ban.fillna("")
sin.loc[sin.conforme_avant, "cause_avant"] = "conforme"
sin["voie_ban_avant"] = sin.voie_ban

  voies :   2000 / 13831 · 1 s
  voies :   4000 / 13831 · 4 s
  voies :   6000 / 13831 · 5 s
  voies :   8000 / 13831 · 7 s
  voies :  10000 / 13831 · 9 s
  voies :  12000 / 13831 · 11 s
  voies :  13831 / 13831 · 13 s
Terminé en 13 s · 14235 réponses en cache · fichier 3.1 Mo


### 1.3 Résultat initial et comparaison avec l'extrait local de S4 (`silver.ref_ban`, US-26)

In [43]:
testees, echecs_avant = len(sin), int((~sin.conforme_avant).sum())
taux_avant = 100 * echecs_avant / testees
print(f"RQ-014 · état initial · non conformes (format + voie à la BAN) : {echecs_avant:,} sur {testees:,} = {taux_avant:.2f} %   (conformité {100 - taux_avant:.3f} %)".replace(",", " "))
print("\nCauses :"); print(sin.cause_avant.value_counts().to_string())

# ---- extrait local de la BAN, définition de S4 (US-26.1) : (code postal, commune, numéro, voie) présent dans silver.ref_ban ----
def maj_pd(s):
    return (s.astype("string").str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii").str.upper()
             .str.replace(r"\s+", " ", regex=True).str.strip())
ban = lire("SELECT numero, voie, code_postal AS cp, commune AS ville FROM silver.ref_ban")
ban["numero"], ban["voie"], ban["ville"] = ban.numero.astype(str), maj_pd(ban.voie), maj_pd(ban.ville)
cles_ban = set(zip(ban.cp, ban.ville, ban.numero, ban.voie))
sin["local_ok"] = [(a, b, c, d) in cles_ban for a, b, c, d in zip(sin.cp, sin.ville, sin.numero.astype(str), sin.voie)]
taux_local = 100 * (~sin.local_ok).mean()
print(f"\nExtrait local (S4, US-26.1) : {taux_local:.2f} % non reconnues   ·   API BAN (voie) : {taux_avant:.2f} % non conformes")
print(pd.crosstab(sin.local_ok.map({True: "local : reconnue", False: "local : non reconnue"}), sin.conforme_avant.map({True: "BAN : conforme", False: "BAN : non conforme"})).to_string())

canal = sin.groupby("canal_saisie").agg(sinistres=("conforme_avant", "size"), non_conformes=("conforme_avant", lambda x: int((~x).sum())))
canal["taux_%"] = (100 * canal.non_conformes / canal.sinistres).round(2)
print("\nPar canal de saisie :"); print(canal.sort_values("non_conformes", ascending=False).to_string())

RQ-014 · état initial · non conformes (format + voie à la BAN) : 82 484 sur 100 000 = 82.48 %   (conformité 17.516 %)

Causes :
cause_avant
BAN : voie différente                                                  69264
conforme                                                               17516
BAN : aucun résultat (voie ou couple code postal / commune inconnu)     4771
BAN : commune différente                                                4498
format : pas de numéro                                                  1982
format : voie ou commune vide                                           1965
BAN : code postal différent                                                4

Extrait local (S4, US-26.1) : 11.87 % non reconnues   ·   API BAN (voie) : 82.48 % non conformes
conforme_avant        BAN : conforme  BAN : non conforme
local_ok                                                
local : non reconnue            1050               10821
local : reconnue               16466               7

**Lecture.** Deux contrôles qui ne mesurent pas la même chose : l'extrait local compare l'adresse **entière** (numéro compris) à 22 856 adresses ; la BAN contrôle que la **voie existe dans la commune et le code postal indiqués**.
Le diagnostic ci-dessous précise pourquoi le niveau « voie » est le bon niveau pour RQ-014 avec ces données.

In [44]:
if ECH_NUMERO:
    base = sin.loc[sin.conforme_avant, ["cp", "ville", "voie", "numero"]].drop_duplicates()
    ech = base.sample(min(ECH_NUMERO, len(base)), random_state=SEED).reset_index(drop=True)
    tup = list(zip(ech.cp, ech.ville, ech.voie, ech.numero))
    assert a_appeler(tup, avec_numero=True) <= MAX_APPELS
    num_res = pd.DataFrame(interroger(tup, controler_numero, "numéros", pas=1000))
    k, n = int(num_res.numero_ok.sum()), len(num_res)
    print(f"Sinistres dont la voie est reconnue : le NUMÉRO existe dans la BAN pour {k} adresses sur {n} = {100 * k / n:.1f} %  (IC 95 % : {wilson(k, n)[0]} à {wilson(k, n)[1]} %)")
if ECH_REFERENTIEL:
    ref = ban.drop_duplicates().sample(min(ECH_REFERENTIEL, len(ban)), random_state=SEED).reset_index(drop=True)
    ref_res = pd.DataFrame(interroger(list(zip(ref.cp, ref.ville, ref.voie, ref.numero)), controler_numero, "référentiel", pas=1000))
    k, n, kv = int(ref_res.numero_ok.sum()), len(ref_res), int(ref_res.voie_ok.sum())
    print(f"L'extrait local lui-même, passé à la BAN ({n} adresses tirées) : voie reconnue {kv}/{n} = {100 * kv / n:.1f} %, numéro reconnu {k}/{n} = {100 * k / n:.1f} %")
    print("→ Les numéros et une grande part des voies du TD sont fictifs : un échec de NUMÉRO à la BAN ne prouve pas une erreur de saisie. RQ-014 est donc mesurée au niveau de la VOIE.")

  numéros :    300 / 300 · 0 s
Sinistres dont la voie est reconnue : le NUMÉRO existe dans la BAN pour 46 adresses sur 300 = 15.3 %  (IC 95 % : 11.7 à 19.8 %)
  référentiel :    100 / 100 · 0 s
L'extrait local lui-même, passé à la BAN (100 adresses tirées) : voie reconnue 24/100 = 24.0 %, numéro reconnu 3/100 = 3.0 %
→ Les numéros et une grande part des voies du TD sont fictifs : un échec de NUMÉRO à la BAN ne prouve pas une erreur de saisie. RQ-014 est donc mesurée au niveau de la VOIE.


### 1.4 Ce que donnerait le certificat avec RQ-014 telle quelle (simulation, rien n'est écrit)

In [45]:
regle = lire("SELECT rule_id, severite, porte, seuil_min FROM dq.regle WHERE rule_id = 'RQ-014'").iloc[0]
DECISION_SANS_14 = """
SELECT m.rule_id, m.table_cible, m.taux_conformite, r.seuil_min
FROM dq.v_derniere_mesure m JOIN dq.regle r USING (rule_id)
WHERE m.batch_id = :lot AND r.severite = 'BLOQUANTE' AND r.porte IN ('B', 'S', 'G')
  AND m.couche = ELT(FIELD(r.porte, 'B', 'S', 'G'), 'BRONZE', 'SILVER', 'GOLD') AND m.rule_id <> 'RQ-014'"""
actuel = lire(DECISION_SANS_14, lot=BATCH)
ok_actuel = int((actuel.taux_conformite.astype(float) >= actuel.seuil_min.astype(float)).sum())
rq14_ok = (100 - taux_avant) >= float(regle.seuil_min)
print(f"Lot de base {BATCH} : {ok_actuel}/{len(actuel)} mesures bloquantes conformes (RQ-014 n'y est pas mesurée).")
print(f"Avec RQ-014 ({regle.severite}, porte {regle.porte}, seuil {float(regle.seuil_min):.0f} %) à {100 - taux_avant:.3f} % : {ok_actuel + int(rq14_ok)}/{len(actuel) + 1} → {'GO' if ok_actuel == len(actuel) and rq14_ok else 'NO_GO'}")

Lot de base L-2026-10-01 : 36/36 mesures bloquantes conformes (RQ-014 n'y est pas mesurée).
Avec RQ-014 (BLOQUANTE, porte S, seuil 100 %) à 17.516 % : 36/37 → NO_GO


---
# Étape 2 · Nettoyer les adresses pour coller au référentiel de la BAN

**Principes**
- On ne **devine pas** : un numéro manquant, une commune ou un code postal incohérent ne se corrigent pas par regex. Seule exception, la règle **K** : une commune absente est complétée depuis le code postal quand le référentiel n'en connaît **qu'une** pour ce code ; l'étape 3 la vérifie ensuite auprès de la BAN.
- Chaque correction est **tracée** (adresse avant, adresse après, règles appliquées).
- Le **retest** de l'étape 3 est le juge : si l'adresse nettoyée n'est pas reconnue alors que l'originale l'était, on garde l'originale (non-régression).
- Le vocabulaire de référence vient de `silver.ref_ban` (extrait local de la BAN).

| Code | Règle | Exemple |
|---|---|---|
| **F** | forme : majuscules, sans accent, `'` et `-` normalisés, ponctuation retirée, espaces simples | `rue de l’Église` → `RUE DE L'EGLISE` |
| **N** | numéro : suffixe séparé (`12B`, `12 BIS`) | `12B RUE X` → `12 B RUE X` |
| **K** | commune absente, complétée depuis le code postal (une seule commune possible dans le référentiel) | `09100` + commune vide → `PAMIERS` |
| **C** | mots collés, coupés au bon endroit | `RUEDE LA PAIX` → `RUE DE LA PAIX` |
| **A** | abréviation du type de voie développée | `R`, `AV`, `CH`, `RTE`… → `RUE`, `AVENUE`, `CHEMIN`, `ROUTE` |
| **T** | faute de frappe dans le type de voie (une seule candidate) | `BOULLEVARD` → `BOULEVARD` |
| **M** | faute de frappe dans un mot (*règle floue*) | `VOLAIRE` → `VOLTAIRE` |
| **V** | voie entière : la voie la plus proche du référentiel si elle est seule à ressembler (*règle floue*) | `PLACE D L'EGLISE` → `PLACE DE L'EGLISE` |

### 2.1 Diagnostic : à quoi ressemblent les erreurs de forme ?

In [46]:
TYPES_VOIE = ["RUE", "AVENUE", "BOULEVARD", "CHEMIN", "QUAI", "PASSAGE", "IMPASSE", "PLACE", "ROUTE", "ALLEE", "COURS", "SQUARE", "VOIE", "SENTIER"]
v = sin.loc[sin.format_ok, "voie"]
premier = v.str.split().str[0]
print("Premier mot de la voie qui n'est pas un type de voie connu :")
print(premier[~premier.isin(TYPES_VOIE)].value_counts().head(15).to_string())
print("\nCaractères autres que lettres, chiffres et espace :", {k: int(n) for k, n in pd.Series(list("".join(v))).loc[lambda s: ~s.str.fullmatch(r"[A-Z0-9 ]")].value_counts().head(8).items()})
print("Espaces doubles ou en bordure :", int(v.str.contains(r"\s{2,}").sum() + (v != v.str.strip()).sum()), " · minuscules :", int(v.str.contains(r"[a-z]").sum()))

Premier mot de la voie qui n'est pas un type de voie connu :
voie
R          1285
AV          293
CH           74
RUEE         66
RE           64
RUUE         58
PASS         54
RU           51
Q            46
RTE          43
IMP          42
PL           40
ALL          35
AVENNUE      16
AVENUEE      15

Caractères autres que lettres, chiffres et espace : {"'": 5971, '-': 1853}
Espaces doubles ou en bordure : 0  · minuscules : 0


### 2.2 Le script de nettoyage (regex + rapprochement du vocabulaire du référentiel)

In [47]:
ABREV_TYPE = {"R": "RUE", "RU": "RUE", "AV": "AVENUE", "AVE": "AVENUE", "AVN": "AVENUE", "BD": "BOULEVARD", "BLD": "BOULEVARD", "BVD": "BOULEVARD",
              "BOUL": "BOULEVARD", "CH": "CHEMIN", "CHE": "CHEMIN", "CHEM": "CHEMIN", "PASS": "PASSAGE", "PAS": "PASSAGE", "PSG": "PASSAGE",
              "IMP": "IMPASSE", "PL": "PLACE", "ALL": "ALLEE", "RTE": "ROUTE", "Q": "QUAI", "SQ": "SQUARE", "CRS": "COURS"}
ABREV_MOT = {"GAL": "GENERAL", "GEN": "GENERAL", "MAL": "MARECHAL", "PDT": "PRESIDENT", "DR": "DOCTEUR", "PR": "PROFESSEUR"}
LIAISONS = ["DES", "DU", "DE", "LA", "LE", "LES", "AU", "AUX", "ET", "SUR", "SOUS"]

def forme(s):
    """Règle F : majuscules, sans accent, ' et - normalisés, pas d'autre ponctuation, espaces simples."""
    s = "" if s is None or (isinstance(s, float) and math.isnan(s)) else str(s)
    s = re.sub(r"[`´‘’ʼ]", "'", s)                                  # apostrophes typographiques (avant la suppression des accents)
    s = re.sub(r"[‐‑‒–—−]", "-", s)                                  # tirets typographiques
    s = s.replace("Œ", "OE").replace("œ", "oe").replace("Æ", "AE").replace("æ", "ae")
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().upper()
    s = re.sub(r"[^A-Z0-9'\- ]", " ", s)                              # ponctuation, symboles
    s = re.sub(r"\s*'\s*", "'", s)                                    # L' EGLISE -> L'EGLISE
    s = re.sub(r"\s*-\s*", "-", s)                                    # SAINT - ANTOINE -> SAINT-ANTOINE
    s = re.sub(r"\bSTE(?=[ -])", "SAINTE", re.sub(r"\bST(?=[ -])", "SAINT", s))
    return re.sub(r"\s+", " ", s).strip()

def decouper(adresse):
    """Règle N : (numéro, suffixe, voie) d'une adresse. '12 B RUE X' -> ('12', 'B', 'RUE X'). Numéro None si l'adresse ne commence pas par un numéro."""
    a = forme(adresse)
    m = re.match(r"^(?:N\s+)?(\d{1,4})\s*(BIS|TER|QUATER|[A-D])?(?=\s|$)\s*(.*)$", a)
    return (m.group(1), m.group(2) or "", m.group(3)) if m else (None, "", a)

class Nettoyeur:
    """Rapproche une voie du référentiel. Renvoie (voie nettoyée, codes des règles appliquées). Les règles M et V sont « floues » (flou=True)."""
    def __init__(self, voies_ref, flou=True):
        self.flou = flou
        self.ref = sorted({forme(v) for v in voies_ref})
        self.ensemble = set(self.ref)
        self.vocab = {w for v in self.ref for w in v.split()} | set(LIAISONS)
        self.contenu = sorted(w for w in self.vocab if len(w) >= 4)
        self.types = sorted(set(TYPES_VOIE) | {v.split()[0] for v in self.ref})
        self.memo = {}

    @staticmethod
    def _unique(mot, candidats, cutoff, marge):
        """La candidate la plus proche (similarité >= cutoff), seulement si elle devance nettement la deuxième : sinon on ne devine pas."""
        r = difflib.get_close_matches(mot, candidats, n=2, cutoff=cutoff)
        if not r:
            return None
        s0 = difflib.SequenceMatcher(None, mot, r[0]).ratio()
        s1 = difflib.SequenceMatcher(None, mot, r[1]).ratio() if len(r) > 1 else 0.0
        return r[0] if s0 - s1 >= marge else None

    def voie(self, brut):
        if brut in self.memo:
            return self.memo[brut]
        f = forme(brut)
        regles = ["F"] if f != brut else []
        t = f.split()
        if not t:
            self.memo[brut] = ("", regles); return self.memo[brut]
        # C : mots collés (RUEDE, AVENUEJEAN, DESARTS)
        if t[0] not in self.types and t[0] not in ABREV_TYPE:
            for ty in sorted(self.types, key=len, reverse=True):
                if t[0].startswith(ty) and len(t[0]) > len(ty) and t[0][len(ty):] in self.vocab:
                    t[0:1] = [ty, t[0][len(ty):]]; regles.append("C"); break
        sortie = t[:1]
        for w in t[1:]:
            if w not in self.vocab:
                for l in sorted(LIAISONS, key=len, reverse=True):
                    if w.startswith(l) and w[len(l):] in self.vocab:
                        sortie += [l, w[len(l):]]; regles.append("C"); break
                else:
                    sortie.append(w)
            else:
                sortie.append(w)
        t = sortie
        # A : abréviation du type de voie ; T : faute de frappe dans le type de voie
        if t[0] in ABREV_TYPE:
            t[0] = ABREV_TYPE[t[0]]; regles.append("A")
        elif t[0] not in self.types:
            c = self._unique(t[0], self.types, 0.75, 0.05)
            if c:
                t[0] = c; regles.append("T")
        out = " ".join(t)
        if self.flou:
            # M : faute de frappe dans un mot de 4 lettres ou plus absent du vocabulaire du référentiel
            for i in range(1, len(t)):
                if t[i] not in self.vocab and len(t[i]) >= 4:
                    c = ABREV_MOT.get(t[i]) or self._unique(t[i], self.contenu, 0.8, 0.05)
                    if c:
                        t[i] = c; regles.append("M")
            out = " ".join(t)
            # V : la voie entière n'est pas dans le référentiel : on prend la plus proche (>= 0,85) si elle est seule à ressembler
            if out not in self.ensemble:
                c = self._unique(out, self.ref, 0.85, 0.03)
                if c:
                    out = c; regles.append("V")
        self.memo[brut] = (out, regles)
        return self.memo[brut]

net = Nettoyeur(ban.voie, flou=NETTOYAGE_FLOU)          # vocabulaire = voies de silver.ref_ban

# règle K : code postal -> commune, seulement si le référentiel n'en connaît qu'une
communes = ban.drop_duplicates(["cp", "ville"]).groupby("cp").ville.agg(list)
commune_unique = {cp: v[0] for cp, v in communes.items() if len(v) == 1}

# ---------- auto-test du nettoyage (sans base, sans réseau) ----------
for entree, attendu in [("R DES ECOLES", "RUE DES ECOLES"), ("AV  DE VERDUN", "AVENUE DE VERDUN"), ("RUEDE LA PAIX", "RUE DE LA PAIX"),
                        ("rue de l’Église", "RUE DE L'EGLISE"), ("CHEMMIN DES VIGNES", "CHEMIN DES VIGNES"), ("RUE SAINT - ANTOINE", "RUE SAINT-ANTOINE"),
                        ("RUE DU 8 MAI 1945", "RUE DU 8 MAI 1945"), ("RUE INCONNUE ZZZ", "RUE INCONNUE ZZZ")]:
    assert net.voie(entree)[0] == attendu, (entree, net.voie(entree))
if NETTOYAGE_FLOU:
    assert net.voie("PLACE D L'EGLISE")[0] == "PLACE DE L'EGLISE" and net.voie("RUE LEON BLUM")[1] == [] and net.voie("RUE JEAN JAUES")[0].endswith("JAURES")
assert decouper("12 B RUE X") == ("12", "B", "RUE X") and decouper("12B RUE X") == ("12", "B", "RUE X") and decouper("12 ALLEE DES LILAS") == ("12", "", "ALLEE DES LILAS")
assert decouper("RUE SANS NUMERO")[0] is None and decouper("N° 7 BIS RUE X") == ("7", "BIS", "RUE X")
print(f"Nettoyage : {len(net.ref)} voies de référence, {len(net.vocab)} mots de vocabulaire · règles floues : {NETTOYAGE_FLOU} · tests réussis.")
print(f"Règle K : {len(commune_unique)} codes postaux sur {len(communes)} n'ont qu'une commune dans le référentiel.")

Nettoyage : 50 voies de référence, 81 mots de vocabulaire · règles floues : True · tests réussis.
Règle K : 172 codes postaux sur 175 n'ont qu'une commune dans le référentiel.


### 2.3 Application aux sinistres

In [48]:
def reconstruire(num, suf, voie):
    return " ".join(x for x in (num, suf, voie) if x)

parts = [decouper(a) for a in sin.adresse_avant]
sin["num_p"], sin["suf_p"] = [p[0] for p in parts], [p[1] for p in parts]
voie_f = pd.Series([p[2] for p in parts], index=sin.index)
cp_ok = sin.cp.str.fullmatch(r"[0-9]{5}")
traite = cp_ok & sin.num_p.notna() & (voie_f != "")           # sans numéro ou sans voie, une adresse n'est pas réparable : on n'invente rien
nettoye = [net.voie(vf) if t else ("", []) for vf, t in zip(voie_f, traite)]
sin["voie_prop"] = [n[0] if t else v for n, t, v in zip(nettoye, traite, sin.voie)]            # voie inchangée pour ce qui n'est pas traité
mod_forme = [forme(a) != a for a in sin.adresse_avant]
sin["adresse_propre"] = [reconstruire(n, s, v) if t else a for n, s, v, t, a in zip(sin.num_p, sin.suf_p, sin.voie_prop, traite, sin.adresse_avant)]

# règle K : commune absente (code postal valide) -> commune unique du référentiel pour ce code postal
sin["ville_prop"] = sin.ville
manque = traite & (sin.ville == "")
sin.loc[manque, "ville_prop"] = sin.loc[manque, "cp"].map(commune_unique).fillna("")
rempli = manque & (sin.ville_prop != "")

sin["nettoyage"] = ["+".join(dict.fromkeys((["F"] if f and t else []) + n[1] + (["N"] if (t and s) else []) + (["K"] if k else [])))
                    for n, t, s, f, k in zip(nettoye, traite, sin.suf_p, mod_forme, rempli)]
sin["format_ok2"] = sin.format_ok | rempli                    # les lignes dont la commune vient d'être complétée peuvent maintenant être contrôlées

change = (sin.adresse_propre != sin.adresse_avant) | (sin.ville_prop != sin.ville)
print(f"Adresses traitées : {int(traite.sum()):,} · modifiées par le nettoyage : {int(change.sum()):,} ({100 * change.mean():.2f} % des sinistres) · communes complétées (K) : {int(rempli.sum()):,}".replace(",", " "))
print("\nRègles appliquées (une ligne peut en cumuler plusieurs) :")
codes = sin.loc[change, "nettoyage"].str.split("+").explode().value_counts()
print(codes.rename_axis("règle").rename("lignes").to_string())
print("\nExemples (avant → après) :")
cols_ex = ["adresse_avant", "adresse_propre", "ville", "ville_prop", "nettoyage"]
ex = sin.loc[change, cols_ex].drop_duplicates("nettoyage")
print(pd.concat([ex, sin.loc[change, cols_ex].sample(min(8, int(change.sum())), random_state=SEED)]).drop_duplicates().head(16).to_string(index=False))
print(f"\nVoie dans l'extrait local avant nettoyage : {100 * sin.loc[traite, 'voie'].isin(net.ensemble).mean():.2f} % · après : {100 * sin.loc[traite, 'voie_prop'].isin(net.ensemble).mean():.2f} %")

Adresses traitées : 98 018 · modifiées par le nettoyage : 5 830 (5.83 % des sinistres) · communes complétées (K) : 1 902

Règles appliquées (une ligne peut en cumuler plusieurs) :
règle
A    1963
K    1902
M     959
T     458
V     391
C     160
F       3

Exemples (avant → après) :
             adresse_avant                 adresse_propre               ville          ville_prop nettoyage
        1 RUE DU COMMERCCE              1 RUE DU COMMERCE              YZEURE              YZEURE         M
            60 RUUE CARNOT                  60 RUE CARNOT            MANOSQUE            MANOSQUE         T
        12 R DU 8 MAI 1945           12 RUE DU 8 MAI 1945     DIGNE-LES-BAINS     DIGNE-LES-BAINS         A
         1 CHEMIN DU STADE              1 CHEMIN DU STADE                                   PARIS         K
       62 PLACE D L'EGLISE           62 PLACE DE L'EGLISE                LAON                LAON         V
        86 AVENUEDES ALPES            86 AVENUE DES ALPES           

---
# Étape 3 · Retester la règle et mettre en quarantaine ce qui ne correspond toujours pas

1. **Retest** : on interroge la BAN avec les voies nettoyées (seules les voies jamais vues déclenchent un appel : le reste vient du cache).
2. **Non-régression** : si l'adresse nettoyée est refusée alors que l'originale était reconnue, on garde l'originale.
3. **Quarantaine** : les lignes encore non conformes sortent de silver et vont dans `dq.rejets` (règle RQ-014, porte S). Rien n'est supprimé : la table de trace `silver.sinistre_r014_trace` garde pour chaque sinistre l'adresse avant, l'adresse après, le verdict et la cause.

In [49]:
cles2 = sin.loc[sin.format_ok2, ["cp", "ville_prop", "voie_prop"]].drop_duplicates().reset_index(drop=True)
couples2 = list(zip(cles2.cp, cles2.ville_prop, cles2.voie_prop))
n2 = a_appeler(couples2)
print(f"{len(couples2):,} voies à contrôler après nettoyage, dont {n2:,} jamais interrogées (≈ {n2 / min(DEBIT_MAX, THREADS / 0.45) / 60:.1f} min)".replace(",", " "))
assert n2 <= MAX_APPELS
resultats2 = interroger(couples2, controler_voie, "voies nettoyées")

r2 = pd.concat([cles2, pd.DataFrame(resultats2)], axis=1).rename(columns={"cause": "cause_ban2", "voie_ok": "voie_ok2", "voie_ban": "voie_ban2"})
sin = sin.merge(r2[["cp", "ville_prop", "voie_prop", "cause_ban2", "voie_ok2", "voie_ban2"]], on=["cp", "ville_prop", "voie_prop"], how="left")
sin["ok_prop"] = sin.format_ok2 & sin.voie_ok2.eq(True)                # adresse nettoyée reconnue
sin["regression"] = sin.conforme_avant & ~sin.ok_prop                   # reconnue avant, refusée après : à examiner (doit rester à zéro)
sin["api_ok"] = sin.ok_prop | sin.conforme_avant
sin["adresse_apres"] = np.where(sin.regression, sin.adresse_avant, sin.adresse_propre)
sin["ville_apres"] = np.where(sin.regression, sin.ville, sin.ville_prop)
sin["voie_ban_finale"] = np.where(sin.ok_prop, sin.voie_ban2, np.where(sin.conforme_avant, sin.voie_ban_avant, sin.voie_ban2))
print(f"Régressions (reconnue avant, refusée après nettoyage) : {int(sin.regression.sum())} → adresse d'origine conservée pour ces lignes.")

10 167 voies à contrôler après nettoyage  dont 0 jamais interrogées (≈ 0.0 min)
  voies nettoyées :   2000 / 10167 · 1 s
  voies nettoyées :   4000 / 10167 · 3 s
  voies nettoyées :   6000 / 10167 · 5 s
  voies nettoyées :   8000 / 10167 · 7 s
  voies nettoyées :  10000 / 10167 · 8 s
  voies nettoyées :  10167 / 10167 · 9 s
Régressions (reconnue avant, refusée après nettoyage) : 18 → adresse d'origine conservée pour ces lignes.


### 3.1 Avant / après nettoyage, vu par la BAN

In [50]:
fmt = ~sin.format_ok2
sin["cause_apres"] = np.where(fmt, sin.cause_avant, np.where(sin.api_ok, "conforme", "BAN : " + sin.cause_ban2.fillna("")))
cmp_ = pd.concat([sin.cause_avant.value_counts(), sin.cause_apres.value_counts()], axis=1, keys=["avant nettoyage", "après nettoyage"]).fillna(0).astype(int)
cmp_["écart"] = cmp_["après nettoyage"] - cmp_["avant nettoyage"]
print(cmp_.to_string())
gain = int(sin.api_ok.sum() - sin.conforme_avant.sum())
print(f"\nConformes à la BAN : {int(sin.conforme_avant.sum()):,} → {int(sin.api_ok.sum()):,} (gain du nettoyage : {gain:,} lignes, soit {100 * gain / len(sin):.2f} point de conformité)".replace(",", " "))

# ---- extrait local après nettoyage (même clé qu'à l'étape 1, avec la voie nettoyée) ----
cles_ref_forme = {(cp, ville, str(n), forme(v)) for cp, ville, n, v in zip(ban.cp, ban.ville, ban.numero, ban.voie)}     # même clé, voies mises en forme
sin["local_ok2"] = [(a, b, str(c), d) in cles_ref_forme for a, b, c, d in zip(sin.cp, sin.ville_prop, sin.num_p, sin.voie_prop)]
print(f"Extrait local (numéro compris) : {int(sin.local_ok.sum()):,} reconnues avant → {int(sin.local_ok2.sum()):,} après nettoyage".replace(",", " "))

                                                                     avant nettoyage  après nettoyage  écart
BAN : voie différente                                                          69264            70745   1481
conforme                                                                       17516            17928    412
BAN : aucun résultat (voie ou couple code postal / commune inconnu)             4771             4711    -60
BAN : commune différente                                                        4498             4571     73
format : pas de numéro                                                          1982             1982      0
format : voie ou commune vide                                                   1965               63  -1902
BAN : code postal différent                                                        4                0     -4

Conformes à la BAN : 17 516 → 17 928 (gain du nettoyage : 412 lignes  soit 0.41 point de conformité)
Extrait local (numéro comp

**Lecture.** Le nettoyage corrige la **forme** (abréviations, fautes de frappe, mots collés) et, avec la règle K, rend **contrôlables** les lignes dont la commune était vide. La BAN, tolérante aux fautes de frappe, reconnaissait déjà beaucoup de lignes mal écrites : le gain sur la conformité reste donc limité. Ce qui reste en erreur est d'une autre nature : la voie **n'existe pas** dans la commune indiquée, ou le couple code postal / commune est incohérent, ou le numéro manque. Aucune regex ne corrige cela.

### 3.2 Critère de quarantaine et volumes

`CRITERE = "API"` met en quarantaine toute ligne non reconnue par la BAN. `CRITERE = "API_OU_LOCAL"` accepte aussi une adresse présente (numéro compris) dans l'extrait local `silver.ref_ban`, qui est l'extrait de la BAN fourni pour le TD.
Le tableau donne les volumes des deux critères avant d'appliquer celui du paramètre.

In [51]:
conf_api = int(sin.api_ok.sum())
conf_hyb = int((sin.api_ok | (sin.format_ok2 & sin.local_ok2)).sum())
vol = pd.DataFrame({"critère": ["API", "API_OU_LOCAL"], "conformes (vers silver/gold)": [conf_api, conf_hyb],
                    "en quarantaine": [len(sin) - conf_api, len(sin) - conf_hyb]})
vol["% en quarantaine"] = (100 * vol["en quarantaine"] / len(sin)).round(2)
vol["appliqué"] = np.where(vol["critère"] == CRITERE, "◄ oui", "")
print(vol.to_string(index=False))

sin["conforme_final"] = sin.api_ok if CRITERE == "API" else (sin.api_ok | (sin.format_ok2 & sin.local_ok2))
sin["cause_finale"] = np.where(sin.conforme_final, np.where(sin.api_ok, "conforme", "conforme (extrait local de la BAN)"),
                               np.where(fmt, sin.cause_avant, sin.cause_apres))
n_q = int((~sin.conforme_final).sum())
print(f"\nCritère appliqué : {CRITERE} → {int(sin.conforme_final.sum()):,} sinistres conformes, {n_q:,} en quarantaine ({100 * n_q / len(sin):.1f} %)".replace(",", " "))
if n_q / len(sin) > 0.5:
    print("⚠ Plus de la moitié des sinistres part en quarantaine. Ce n'est pas un défaut du nettoyage : avec les données du TD, la plupart des voies")
    print("  (génériques, sur des communes réelles) n'existent pas dans la vraie BAN. Pour un jeu cohérent avec l'extrait de S4, passer CRITERE à \"API_OU_LOCAL\".")
print("\nCauses de la quarantaine :"); print(sin.loc[~sin.conforme_final, "cause_finale"].value_counts().to_string())
canal = sin.groupby("canal_saisie").agg(sinistres=("conforme_final", "size"), en_quarantaine=("conforme_final", lambda x: int((~x).sum())))
canal["%"] = (100 * canal.en_quarantaine / canal.sinistres).round(2)
print("\nQuarantaine par canal de saisie :"); print(canal.sort_values("en_quarantaine", ascending=False).to_string())

     critère  conformes (vers silver/gold)  en quarantaine  % en quarantaine appliqué
         API                         17928           82072             82.07         
API_OU_LOCAL                         94362            5638              5.64    ◄ oui

Critère appliqué : API_OU_LOCAL → 94 362 sinistres conformes  5 638 en quarantaine (5.6 %)

Causes de la quarantaine :
cause_finale
format : pas de numéro                                                 1982
BAN : voie différente                                                  1462
BAN : aucun résultat (voie ou couple code postal / commune inconnu)    1216
BAN : commune différente                                                915
format : voie ou commune vide                                            63

Quarantaine par canal de saisie :
                    sinistres  en_quarantaine      %
canal_saisie                                        
ressaisie_manuelle      30000            3845  12.82
portail_assure          40000      

### 3.3 Écriture : silver, quarantaine, trace

Rejouable : à chaque exécution, les objets du lot `R14` sont vidés puis réécrits. Le lot de base n'est pas touché.

In [52]:
DDL_R14 = [
 """CREATE TABLE IF NOT EXISTS silver.sinistre_r014_trace (
  silver_row_id BIGINT PRIMARY KEY, sinistre_id VARCHAR(12) NOT NULL, lot VARCHAR(20) NOT NULL,
  adresse_avant VARCHAR(160), adresse_apres VARCHAR(160), ville_avant VARCHAR(80), ville_apres VARCHAR(80), nettoyage VARCHAR(20),
  conforme_avant TINYINT NOT NULL, conforme TINYINT NOT NULL, cause_avant VARCHAR(255), cause VARCHAR(255), voie_ban VARCHAR(160))""",
 """CREATE TABLE IF NOT EXISTS silver.sinistre_r014 (
  silver_row_id BIGINT PRIMARY KEY, bronze_row_id BIGINT NOT NULL, sinistre_id VARCHAR(12) NOT NULL, contrat_id VARCHAR(12),
  date_survenance DATE, date_declaration DATE, type_sinistre VARCHAR(40), montant_estime DECIMAL(12,2),
  adresse_normalisee VARCHAR(160), cp VARCHAR(10), ville_normalisee VARCHAR(80), canal_saisie VARCHAR(30),
  adresse_avant VARCHAR(160), nettoyage VARCHAR(20), voie_ban VARCHAR(160),
  _batch_id VARCHAR(20) NOT NULL, _processed_at DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
  UNIQUE KEY uq_bronze (bronze_row_id), KEY idx_contrat (contrat_id))"""]

trace = pd.DataFrame({
    "silver_row_id": sin.silver_row_id.astype(int), "sinistre_id": sin.sinistre_id, "lot": LOT,
    "adresse_avant": sin.adresse_avant.str[:160], "adresse_apres": sin.adresse_apres.str[:160], "ville_avant": sin.ville.str[:80], "ville_apres": sin.ville_apres.str[:80],
    "nettoyage": sin.nettoyage.str[:20],
    "conforme_avant": sin.conforme_avant.astype(int), "conforme": sin.conforme_final.astype(int),
    "cause_avant": sin.cause_avant.str[:255], "cause": sin.cause_finale.str[:255],
    "voie_ban": sin.voie_ban_finale.where(sin.api_ok & sin.conforme_final).astype("object").str[:160]})

if ECRIRE:
    for ddl in DDL_R14:
        executer(ddl)
    executer("DELETE FROM silver.sinistre_r014_trace"); executer("DELETE FROM silver.sinistre_r014")
    executer("DELETE FROM dq.rejets WHERE batch_id = :lot", lot=LOT)
    ecrire(trace, "silver", "sinistre_r014_trace")
    executer("""INSERT INTO silver.sinistre_r014 (silver_row_id, bronze_row_id, sinistre_id, contrat_id, date_survenance, date_declaration, type_sinistre, montant_estime,
                  adresse_normalisee, cp, ville_normalisee, canal_saisie, adresse_avant, nettoyage, voie_ban, _batch_id)
                SELECT s.silver_row_id, s.bronze_row_id, s.sinistre_id, s.contrat_id, s.date_survenance, s.date_declaration, s.type_sinistre, s.montant_estime,
                       t.adresse_apres, s.cp, t.ville_apres, s.canal_saisie, t.adresse_avant, t.nettoyage, t.voie_ban, :lot
                FROM silver.sinistre s JOIN silver.sinistre_r014_trace t ON t.silver_row_id = s.silver_row_id WHERE t.conforme = 1""", lot=LOT)
    # quarantaine : les rejets antérieurs du lot de base (porte S, sinistre) restent rejetés ; s'y ajoutent les lignes RQ-014
    executer("""INSERT INTO dq.rejets (batch_id, entite, source_systeme, bronze_row_id, rule_id, porte, motif)
                SELECT :lot, entite, source_systeme, bronze_row_id, rule_id, porte, motif FROM dq.rejets WHERE batch_id = :b AND entite = 'sinistre' AND porte = 'S'""", lot=LOT, b=BATCH)
    executer("""INSERT INTO dq.rejets (batch_id, entite, source_systeme, bronze_row_id, rule_id, porte, motif)
                SELECT :lot, 'sinistre', 'crm', s.bronze_row_id, 'RQ-014', 'S', LEFT(t.cause, 255)
                FROM silver.sinistre s JOIN silver.sinistre_r014_trace t ON t.silver_row_id = s.silver_row_id WHERE t.conforme = 0""", lot=LOT)
    print("silver.sinistre_r014_trace :", lire("SELECT COUNT(*) AS n FROM silver.sinistre_r014_trace").n[0], "lignes (tous les sinistres, avant / après / verdict)")
    print("silver.sinistre_r014       :", lire("SELECT COUNT(*) AS n FROM silver.sinistre_r014").n[0], "lignes (conformes RQ-014)")
    print(lire("SELECT entite, rule_id, porte, COUNT(*) AS lignes FROM dq.rejets WHERE batch_id = :lot GROUP BY entite, rule_id, porte ORDER BY rule_id", lot=LOT).to_string(index=False))
else:
    print(f"ECRIRE = False : {int(sin.conforme_final.sum()):,} lignes iraient dans silver.sinistre_r014, {n_q:,} dans dq.rejets (RQ-014).".replace(",", " "))

silver.sinistre_r014_trace : 100000 lignes (tous les sinistres, avant / après / verdict)
silver.sinistre_r014       : 94362 lignes (conformes RQ-014)
  entite rule_id porte  lignes
sinistre  RQ-014     S    5638


### 3.4 Mesure de RQ-014 relue depuis la base

La mesure ne se fait pas sur les données en mémoire mais sur ce qui est **réellement enregistré** dans `silver.sinistre_r014` : on relit les adresses stockées, on refait le contrôle de format, et on retrouve le verdict de la BAN (réponses en cache) pour la voie stockée. Attendu : 0 échec.

In [53]:
def cle_verdict(cp, ville, voie):
    return (cp, ville, forme(voie))

# verdict BAN par (cp, commune, voie normalisée) : étape 1 (voies d'origine) et étape 3 (voies nettoyées)
verdict = {}
for (cp, ville, voie), r in zip(couples, resultats):
    verdict[cle_verdict(cp, ville, voie)] = verdict.get(cle_verdict(cp, ville, voie), False) or bool(r["voie_ok"])
for (cp, ville, voie), r in zip(couples2, resultats2):
    verdict[cle_verdict(cp, ville, voie)] = verdict.get(cle_verdict(cp, ville, voie), False) or bool(r["voie_ok"])

def statut_rq014(df):
    """Contrôle RQ-014 d'adresses relues depuis la base : format + voie reconnue (BAN, cache) ou, selon CRITERE, présente dans l'extrait local."""
    p = [decouper(a) for a in df.adresse_normalisee]
    num, voie = [x[0] for x in p], [x[2] for x in p]
    fmt_ok = [bool(re.fullmatch(r"[0-9]{5}", cp)) and n is not None and v != "" and vl != "" for cp, n, v, vl in zip(df.cp, num, voie, df.ville)]
    api = [verdict.get(cle_verdict(cp, vl, v), False) for cp, vl, v in zip(df.cp, df.ville, voie)]
    loc = [(cp, vl, n, forme(v)) in cles_ref_forme for cp, vl, n, v in zip(df.cp, df.ville, num, voie)] if CRITERE == "API_OU_LOCAL" else [False] * len(df)
    return [f and (a or l) for f, a, l in zip(fmt_ok, api, loc)]

if ECRIRE:
    relu = lire("SELECT silver_row_id, cp, ville_normalisee AS ville, adresse_normalisee FROM silver.sinistre_r014")
    ok = statut_rq014(relu)
    rq14_testees, rq14_echecs = len(relu), int(len(relu) - sum(ok))
    print(f"RQ-014 sur silver.sinistre_r014 (relu depuis la base) : {rq14_echecs} non conformes sur {rq14_testees:,} = conformité {100 * (rq14_testees - rq14_echecs) / rq14_testees:.3f} %".replace(",", " "))
    # contre-épreuve : la même fonction appliquée aux lignes mises en quarantaine doit les refuser
    q = sin.loc[~sin.conforme_final, ["cp", "ville", "adresse_apres"]].rename(columns={"adresse_apres": "adresse_normalisee"})
    print(f"Contre-épreuve : {int(sum(statut_rq014(q)))} ligne(s) de la quarantaine passeraient le contrôle (attendu : 0).")
else:
    rq14_testees, rq14_echecs = int(sin.conforme_final.sum()), 0
    print("ECRIRE = False : mesure non relue en base.")

RQ-014 sur silver.sinistre_r014 (relu depuis la base) : 0 non conformes sur 94 362 = conformité 100.000 %
Contre-épreuve : 0 ligne(s) de la quarantaine passeraient le contrôle (attendu : 0).


---
# Étape 4 · Repasser en gold, mesurer, certifier, publier

1. **Gold** : `gold.sinistre_certifie_r014` = les sinistres de `silver.sinistre_r014` dont le contrat est dans `gold.contrat_certifie` et dont la survenance est dans la période de garantie (RQ-017, comme en S4). Clients et contrats ne changent pas : on réutilise le gold du lot de base.
2. **Mesures** du lot `R14` : les mesures du lot de base sont reprises ; celles qui portent sur les sinistres sont **recalculées** sur les nouvelles tables ; RQ-014 est ajoutée (SILVER, sinistre).
3. **Certificat** : même décision que S5 (toutes les règles bloquantes des portes B, S, G conformes sur la couche de leur porte), empreinte SHA-256 du gold du lot `R14`.
4. **Publication** vers `crm_cible.sinistre_r014`, avec les refus de S5 (pas de certificat, pas GO, gold modifié) et la réconciliation RQ-025.

### 4.1 Gold

In [54]:
if ECRIRE:
    executer("CREATE TABLE IF NOT EXISTS gold.sinistre_certifie_r014 LIKE gold.sinistre_certifie")
    executer("DELETE FROM gold.sinistre_certifie_r014")
    executer("""INSERT INTO gold.sinistre_certifie_r014 (sinistre_id, contrat_id, date_survenance, date_declaration, type_sinistre, montant_estime,
                  adresse_normalisee, cp, ville_normalisee, canal_saisie, bronze_row_id, silver_row_id, _batch_id)
                SELECT s.sinistre_id, s.contrat_id, s.date_survenance, s.date_declaration, s.type_sinistre, s.montant_estime,
                       s.adresse_normalisee, s.cp, s.ville_normalisee, s.canal_saisie, s.bronze_row_id, s.silver_row_id, :lot
                FROM silver.sinistre_r014 s JOIN gold.contrat_certifie c ON c.contrat_id = s.contrat_id
                WHERE s.date_survenance BETWEEN c.date_debut AND c.date_fin""", lot=LOT)
    executer("DELETE FROM dq.rejets WHERE batch_id = :lot AND porte = 'G'", lot=LOT)
    executer("""INSERT INTO dq.rejets (batch_id, entite, source_systeme, bronze_row_id, rule_id, porte, motif)
                SELECT :lot, 'sinistre', 'crm', s.bronze_row_id, 'RQ-017', 'G',
                       CASE WHEN c.contrat_id IS NULL THEN 'contrat absent du gold (rejeté en amont)' ELSE 'survenance hors période de garantie' END
                FROM silver.sinistre_r014 s LEFT JOIN gold.contrat_certifie c ON c.contrat_id = s.contrat_id
                WHERE c.contrat_id IS NULL OR NOT COALESCE(s.date_survenance BETWEEN c.date_debut AND c.date_fin, 0)""", lot=LOT)
    n_gold = int(lire("SELECT COUNT(*) AS n FROM gold.sinistre_certifie_r014").n[0])
    n_silver14 = int(lire("SELECT COUNT(*) AS n FROM silver.sinistre_r014").n[0])
    n_rej_g = int(lire("SELECT COUNT(*) AS n FROM dq.rejets WHERE batch_id = :lot AND porte = 'G'", lot=LOT).n[0])
    n_commun = int(lire("SELECT COUNT(*) AS n FROM gold.sinistre_certifie g JOIN silver.sinistre_r014 s ON s.sinistre_id = g.sinistre_id").n[0])
    print(f"silver.sinistre_r014 {n_silver14:,} = gold.sinistre_certifie_r014 {n_gold:,} + rejets de la porte G (RQ-017) {n_rej_g:,}".replace(",", " "))
    print(f"Contrôle croisé avec le gold du lot de base : {n_commun:,} sinistres communs, {'identique' if n_commun == n_gold else 'ÉCART de ' + str(n_gold - n_commun)}".replace(",", " "))
    assert n_silver14 == n_gold + n_rej_g, "silver != gold + rejets G"
else:
    print("ECRIRE = False : étape 4 non exécutée.")

silver.sinistre_r014 94 362 = gold.sinistre_certifie_r014 91 101 + rejets de la porte G (RQ-017) 3 261
Contrôle croisé avec le gold du lot de base : 91 101 sinistres communs  identique


### 4.2 Mesures du lot `R14`

Reprises : tout ce qui ne dépend pas des adresses de sinistre (bronze, clients, contrats…). Recalculées : les mesures SILVER et GOLD sur les sinistres (RQ-017, RQ-020, RQ-023, RQ-024). Ajoutée : RQ-014 (SILVER, sinistre). Une mesure de sinistre du lot de base que ce notebook ne sait pas recalculer est reprise telle quelle et signalée.

In [55]:
def mesurer_lot(couche, rule_id, table, testees, echecs):
    """Équivalent de aq.mesurer pour le lot R14 (aq.mesurer écrit toujours dans le lot de base)."""
    p = {"b": LOT, "c": couche, "r": rule_id, "t": table, "n": int(testees), "f": int(echecs)}
    executer("DELETE FROM dq.dq_history WHERE batch_id = :b AND couche = :c AND rule_id = :r AND table_cible = :t", **p)
    executer("INSERT INTO dq.dq_history (batch_id, couche, rule_id, table_cible, nb_testees, nb_echecs) VALUES (:b, :c, :r, :t, :n, :f)", **p)

RECALCUL = {   # (couche, règle, table) -> requête qui renvoie (lignes testées, lignes en échec) sur les nouvelles tables
 ("SILVER", "RQ-017", "sinistre"): """SELECT COUNT(*), COALESCE(SUM(NOT (s.date_survenance BETWEEN c.date_debut AND c.date_fin)), 0)
      FROM silver.sinistre_r014 s JOIN silver.contrat c ON c.contrat_id = s.contrat_id""",
 ("SILVER", "RQ-020", "sinistre"): "SELECT COUNT(*), COUNT(*) - COUNT(DISTINCT sinistre_id) + COALESCE(SUM(sinistre_id IS NULL), 0) FROM silver.sinistre_r014",
 ("SILVER", "RQ-023", "sinistre"): """SELECT COUNT(*), COALESCE(SUM(NOT EXISTS (SELECT 1 FROM bronze.crm_sinistre b WHERE b.bronze_row_id = s.bronze_row_id)), 0) FROM silver.sinistre_r014 s""",
 ("GOLD", "RQ-017", "sinistre_certifie"): """SELECT COUNT(*), COALESCE(SUM(NOT (s.date_survenance BETWEEN c.date_debut AND c.date_fin)), 0)
      FROM gold.sinistre_certifie_r014 s JOIN gold.contrat_certifie c ON c.contrat_id = s.contrat_id""",
 ("GOLD", "RQ-020", "sinistre_certifie"): "SELECT COUNT(*), COUNT(*) - COUNT(DISTINCT sinistre_id) FROM gold.sinistre_certifie_r014",
 ("GOLD", "RQ-023", "sinistre_certifie"): """SELECT COUNT(*), COALESCE(SUM(NOT EXISTS (SELECT 1 FROM bronze.crm_sinistre b WHERE b.bronze_row_id = g.bronze_row_id)), 0) FROM gold.sinistre_certifie_r014 g"""}
TABLES_SINISTRE = ("sinistre", "sinistre_certifie")

if ECRIRE:
    base_m = lire("SELECT couche, rule_id, table_cible, nb_testees, nb_echecs FROM dq.v_derniere_mesure WHERE batch_id = :b", b=BATCH)
    sinistre_m = base_m.couche.isin(["SILVER", "GOLD", "CRM"]) & base_m.table_cible.isin(TABLES_SINISTRE)
    executer("DELETE FROM dq.dq_history WHERE batch_id = :lot", lot=LOT)
    journal = []
    for r in base_m[~sinistre_m].itertuples():                                         # 1. reprises
        mesurer_lot(r.couche, r.rule_id, r.table_cible, r.nb_testees, r.nb_echecs)
        journal.append((r.couche, r.rule_id, r.table_cible, int(r.nb_testees), int(r.nb_echecs), "reprise du lot de base"))
    # RQ-024 : silver + rejets de la porte S = bronze (la mesure de base fixe le « bronze » : nb_testees)
    ecart_base = ecart_nouveau = None
    for r in base_m[sinistre_m].itertuples():                                          # 2. mesures de sinistre : recalcul si on sait, sinon reprise signalée
        cle = (r.couche, r.rule_id, r.table_cible)
        if cle in RECALCUL:
            n, e = lire(RECALCUL[cle]).iloc[0]; src = "recalculée"
        elif cle == ("SILVER", "RQ-024", "sinistre"):
            rej_b = int(lire("SELECT COUNT(*) AS n FROM dq.rejets WHERE batch_id = :b AND entite = 'sinistre' AND porte = 'S'", b=BATCH).n[0])
            rej_n = int(lire("SELECT COUNT(*) AS n FROM dq.rejets WHERE batch_id = :lot AND entite = 'sinistre' AND porte = 'S'", lot=LOT).n[0])
            sil_b = int(lire("SELECT COUNT(*) AS n FROM silver.sinistre").n[0]); sil_n = int(lire("SELECT COUNT(*) AS n FROM silver.sinistre_r014").n[0])
            ecart_base, ecart_nouveau = sil_b + rej_b - int(r.nb_testees), sil_n + rej_n - int(r.nb_testees)
            n, e = int(r.nb_testees), abs(ecart_nouveau - ecart_base); src = "recalculée"
        elif cle[0] == "CRM":
            continue                                                                    # RQ-025 du lot R14 : mesurée à la publication (4.4)
        else:
            n, e = r.nb_testees, r.nb_echecs; src = "REPRISE du lot de base (non recalculable ici)"
        mesurer_lot(r.couche, r.rule_id, r.table_cible, n, e)
        journal.append((r.couche, r.rule_id, r.table_cible, int(n), int(e), src))
    mesurer_lot("SILVER", "RQ-014", "sinistre", rq14_testees, rq14_echecs)           # 3. la nouvelle mesure
    journal.append(("SILVER", "RQ-014", "sinistre", rq14_testees, rq14_echecs, "NOUVELLE (relue depuis la base)"))
    J = pd.DataFrame(journal, columns=["couche", "règle", "table", "testées", "échecs", "origine"])
    print(J["origine"].value_counts().to_string())
    print("\nMesures sur les sinistres :"); print(J[J.table.isin(TABLES_SINISTRE)].sort_values(["couche", "règle"]).to_string(index=False))
    if ecart_base is not None:
        print(f"\nRQ-024 (conservation) : écart du lot de base {ecart_base}, du lot R14 {ecart_nouveau} (silver + rejets S - bronze).")
else:
    print("ECRIRE = False : étape 4 non exécutée.")

origine
reprise du lot de base             72
recalculée                          7
NOUVELLE (relue depuis la base)     1

Mesures sur les sinistres :
couche  règle             table  testées  échecs                         origine
  GOLD RQ-017 sinistre_certifie    91101       0                      recalculée
  GOLD RQ-020 sinistre_certifie    91101       0                      recalculée
  GOLD RQ-023 sinistre_certifie    91101       0                      recalculée
SILVER RQ-014          sinistre    94362       0 NOUVELLE (relue depuis la base)
SILVER RQ-017          sinistre    93787    1119                      recalculée
SILVER RQ-020          sinistre    94362       0                      recalculée
SILVER RQ-023          sinistre    94362       0                      recalculée
SILVER RQ-024          sinistre   100000       0                      recalculée

RQ-024 (conservation) : écart du lot de base 0, du lot R14 0 (silver + rejets S - bronze).


### 4.3 Certificat de qualité

In [56]:
DECISION = """
SELECT CASE WHEN SUM(m.taux_conformite < r.seuil_min) = 0 THEN 'GO' ELSE 'NO_GO' END AS decision,
       COUNT(*) AS nb_testees, COALESCE(SUM(m.taux_conformite >= r.seuil_min), 0) AS nb_ok
FROM dq.v_derniere_mesure m JOIN dq.regle r USING (rule_id)
WHERE m.batch_id = :lot AND r.severite = 'BLOQUANTE' AND r.porte IN ('B', 'S', 'G')
  AND m.couche = ELT(FIELD(r.porte, 'B', 'S', 'G'), 'BRONZE', 'SILVER', 'GOLD')"""
EN_ECHEC = """
SELECT m.rule_id, m.table_cible, m.taux_conformite, r.seuil_min
FROM dq.v_derniere_mesure m JOIN dq.regle r USING (rule_id)
WHERE m.batch_id = :lot AND r.severite = 'BLOQUANTE' AND r.porte IN ('B', 'S', 'G')
  AND m.couche = ELT(FIELD(r.porte, 'B', 'S', 'G'), 'BRONZE', 'SILVER', 'GOLD') AND m.taux_conformite < r.seuil_min"""
GOLD_R14 = ["client_unique", "contrat_certifie", "sinistre_certifie_r014"]        # clients et contrats : ceux du lot de base

def empreinte_gold_r14():
    """Même algorithme que aq.empreinte_gold (SHA-256, colonnes techniques exclues, lignes triées), sur le gold du lot R14."""
    h = hashlib.sha256()
    for t in GOLD_R14:
        df = lire(f"SELECT * FROM gold.{t}")
        df = df[[c for c in df.columns if not c.startswith("_")]].astype(str)
        df = df.sort_values(list(df.columns)).reset_index(drop=True)
        h.update(t.encode()); h.update(pd.util.hash_pandas_object(df, index=False).values.tobytes())
    return h.hexdigest()

def emettre_r14(signataire="Data Owner"):
    d = lire(DECISION, lot=LOT).iloc[0]
    ko = lire(EN_ECHEC, lot=LOT)
    commentaire = ("RQ-014 (BAN) incluse ; clients et contrats du lot " + BATCH if d.decision == "GO"
                   else "Regles en echec : " + ", ".join(f"{r.rule_id} ({r.table_cible})" for r in ko.itertuples()))
    executer("DELETE FROM dq.certificat_qualite WHERE certificat_id = :i", i=CERT)
    executer("""INSERT INTO dq.certificat_qualite (certificat_id, batch_id, nb_bloquantes_testees, nb_bloquantes_ok, empreinte_gold, decision, signataire, commentaire)
                VALUES (:i, :b, :nt, :nok, :e, :d, :s, :c)""",
             i=CERT, b=LOT, nt=int(d.nb_testees), nok=int(d.nb_ok), e=empreinte_gold_r14(), d=d.decision, s=signataire, c=commentaire[:500])
    return ko

if ECRIRE:
    ko = emettre_r14()
    print(lire("SELECT certificat_id, decision, nb_bloquantes_ok, nb_bloquantes_testees, SUBSTR(empreinte_gold, 1, 16) AS empreinte_debut, signataire, commentaire FROM dq.certificat_qualite WHERE certificat_id IN (:a, :b) ORDER BY certificat_id",
               a=CERT, b=f"CERT-{BATCH}").to_string(index=False))
    print("Mesures bloquantes sous leur seuil :", ko.rule_id.tolist() or "aucune")
else:
    print("ECRIRE = False : étape 4 non exécutée.")

        certificat_id decision  nb_bloquantes_ok  nb_bloquantes_testees  empreinte_debut signataire                                                    commentaire
    CERT-L-2026-10-01       GO                36                     36 07667ef351b7207c Data Owner                                Aucune regle bloquante en echec
CERT-L-2026-10-01-R14       GO                37                     37 7886cff8adb89701 Data Owner RQ-014 (BAN) incluse ; clients et contrats du lot L-2026-10-01
Mesures bloquantes sous leur seuil : aucune


### 4.4 Publication vers la nouvelle table cible `crm_cible.sinistre_r014`

In [57]:
class Refus(Exception):
    pass

def colonnes_gold(table, techniques=False):
    cols = list(lire(f"SELECT * FROM gold.{table} LIMIT 0").columns)
    return cols if techniques else [c for c in cols if not c.startswith("_")]

def publier_r14(certificat_id):
    """Trois refus (aucun certificat, décision NO_GO, gold modifié depuis l'émission), puis chargement en une seule transaction et réconciliation RQ-025."""
    cert = lire("SELECT certificat_id, decision, empreinte_gold FROM dq.certificat_qualite WHERE certificat_id = :i", i=certificat_id)
    if cert.empty:
        raise Refus("REFUS : aucun certificat de qualité")
    cert = cert.iloc[0]
    if cert.decision != "GO":
        raise Refus(f"REFUS : décision {cert.decision} sur {cert.certificat_id}")
    if cert.empreinte_gold != empreinte_gold_r14():
        raise Refus("REFUS : le gold a changé depuis l'émission du certificat")
    executer("CREATE TABLE IF NOT EXISTS crm_cible.sinistre_r014 LIKE gold.sinistre_certifie_r014")
    cols = ", ".join(colonnes_gold("sinistre_certifie_r014", techniques=True))
    with eng().begin() as cx:                                            # une seule transaction : rien de partiel
        cx.execute(text("DELETE FROM crm_cible.sinistre_r014"))
        cx.execute(text(f"INSERT INTO crm_cible.sinistre_r014 ({cols}) SELECT {cols} FROM gold.sinistre_certifie_r014"))
    crc = "SUM(CRC32(CONCAT_WS('|', " + ", ".join(colonnes_gold("sinistre_certifie_r014")) + ")))"
    g = lire(f"SELECT COUNT(*) AS n, COALESCE({crc}, 0) AS crc FROM gold.sinistre_certifie_r014").iloc[0]
    c = lire(f"SELECT COUNT(*) AS n, COALESCE({crc}, 0) AS crc FROM crm_cible.sinistre_r014").iloc[0]
    mesurer_lot("CRM", "RQ-025", "sinistre", int(g.n), abs(int(g.n) - int(c.n)) + (1 if g.crc != c.crc else 0))
    return f"PUBLIÉ : {cert.certificat_id}"

if ECRIRE:
    for libelle, cid in [("Refus 1 · certificat absent", "CERT-INEXISTANT"), ("Publication du lot R14", CERT)]:
        try:
            print(f"{libelle:<30} -> {publier_r14(cid)}")
        except Refus as r:
            print(f"{libelle:<30} -> {r}")
    print(lire("SELECT rule_id, table_cible, nb_testees, nb_echecs, taux_conformite FROM dq.dq_history WHERE batch_id = :lot AND couche = 'CRM'", lot=LOT).to_string(index=False))
else:
    print("ECRIRE = False : étape 4 non exécutée.")

Refus 1 · certificat absent    -> REFUS : aucun certificat de qualité
Publication du lot R14         -> PUBLIÉ : CERT-L-2026-10-01-R14
rule_id table_cible  nb_testees  nb_echecs taux_conformite
 RQ-025      client      120480          0         100.000
 RQ-025     contrat       95866          0         100.000
 RQ-025    sinistre       91101          0         100.000


### 4.5 Bilan : lot de base et lot `R14`

In [58]:
def compter(requete, **p):
    try:
        return int(lire(requete, **p).iloc[0, 0])
    except Exception:
        return None

if ECRIRE:
    bilan = pd.DataFrame([
        ("Sinistres en bronze", compter("SELECT COUNT(*) FROM bronze.crm_sinistre"), compter("SELECT COUNT(*) FROM bronze.crm_sinistre")),
        ("Quarantaine porte S (tous motifs)", compter("SELECT COUNT(*) FROM dq.rejets WHERE batch_id = :b AND entite = 'sinistre' AND porte = 'S'", b=BATCH),
                                              compter("SELECT COUNT(*) FROM dq.rejets WHERE batch_id = :b AND entite = 'sinistre' AND porte = 'S'", b=LOT)),
        ("   dont RQ-014 (adresse)", 0, compter("SELECT COUNT(*) FROM dq.rejets WHERE batch_id = :b AND rule_id = 'RQ-014'", b=LOT)),
        ("Sinistres en silver", compter("SELECT COUNT(*) FROM silver.sinistre"), compter("SELECT COUNT(*) FROM silver.sinistre_r014")),
        ("Quarantaine porte G (RQ-017)", compter("SELECT COUNT(*) FROM dq.rejets WHERE batch_id = :b AND entite = 'sinistre' AND porte = 'G'", b=BATCH),
                                         compter("SELECT COUNT(*) FROM dq.rejets WHERE batch_id = :b AND entite = 'sinistre' AND porte = 'G'", b=LOT)),
        ("Sinistres en gold", compter("SELECT COUNT(*) FROM gold.sinistre_certifie"), compter("SELECT COUNT(*) FROM gold.sinistre_certifie_r014")),
        ("Sinistres dans le CRM cible", compter("SELECT COUNT(*) FROM crm_cible.sinistre"), compter("SELECT COUNT(*) FROM crm_cible.sinistre_r014"))],
        columns=["", f"lot de base {BATCH} (crm_cible.sinistre)", f"lot {LOT} (crm_cible.sinistre_r014)"])
    print(bilan.to_string(index=False))
    cert = lire("SELECT certificat_id, decision, nb_bloquantes_ok, nb_bloquantes_testees FROM dq.certificat_qualite WHERE certificat_id IN (:a, :b) ORDER BY certificat_id", a=CERT, b=f"CERT-{BATCH}")
    print("\nCertificats :"); print(cert.to_string(index=False))
    print(f"\nRQ-014 : {100 - taux_avant:.2f} % de conformité à la BAN en silver au départ → 100 % sur le lot {LOT} (les lignes non conformes sont en quarantaine, avec leur cause).")
else:
    print("ECRIRE = False : étape 4 non exécutée.")

                                   lot de base L-2026-10-01 (crm_cible.sinistre)  lot L-2026-10-01-R14 (crm_cible.sinistre_r014)
              Sinistres en bronze                                         100000                                          100000
Quarantaine porte S (tous motifs)                                              0                                            5638
            dont RQ-014 (adresse)                                              0                                            5638
              Sinistres en silver                                         100000                                           94362
     Quarantaine porte G (RQ-017)                                           3468                                            3261
                Sinistres en gold                                          96532                                           91101
      Sinistres dans le CRM cible                                          96532                 

---
## Preuves à conserver (`reports/`)
Les fichiers `.csv` sont dans `.gitignore` : ils restent sur le poste, comme preuves de l'exécution.

In [59]:
os.makedirs(f"{RACINE}/reports", exist_ok=True)
resume = pd.DataFrame([{
    "lot_base": BATCH, "lot_r14": LOT, "mesure_le": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), "api": URL_BAN, "critere": CRITERE,
    "seuil_similarite_voie": SEUIL_VOIE, "nettoyage_flou": NETTOYAGE_FLOU, "sinistres": len(sin),
    "conformes_ban_avant": int(sin.conforme_avant.sum()), "conformes_ban_apres_nettoyage": int(sin.api_ok.sum()),
    "lignes_modifiees_par_nettoyage": int(((sin.adresse_apres != sin.adresse_avant) | (sin.ville_apres != sin.ville)).sum()), "communes_completees": int(sin.nettoyage.str.contains("K").sum()), "regressions": int(sin.regression.sum()),
    "conformes_retenus": int(sin.conforme_final.sum()), "en_quarantaine": int((~sin.conforme_final).sum()),
    "ecrit_en_base": bool(ECRIRE)}])
resume.to_csv(f"{RACINE}/reports/rq014_lot_resume.csv", index=False, encoding="utf-8")
sin.loc[~sin.conforme_final, ["sinistre_id", "canal_saisie", "cp", "ville", "adresse_avant", "adresse_apres", "cause_finale"]].to_csv(f"{RACINE}/reports/rq014_lot_quarantaine.csv", index=False, encoding="utf-8")
sin.loc[(sin.adresse_apres != sin.adresse_avant) | (sin.ville_apres != sin.ville), ["sinistre_id", "adresse_avant", "adresse_apres", "ville", "ville_apres", "nettoyage", "conforme_avant", "conforme_final"]].to_csv(f"{RACINE}/reports/rq014_lot_nettoyage.csv", index=False, encoding="utf-8")
print(resume.T.to_string(header=False))
print("\nFichiers : reports/rq014_lot_resume.csv · rq014_lot_quarantaine.csv · rq014_lot_nettoyage.csv · ban_cache.json")

lot_base                                                  L-2026-10-01
lot_r14                                               L-2026-10-01-R14
mesure_le                                             2026-11-20 15:21
api                             https://data.geopf.fr/geocodage/search
critere                                                   API_OU_LOCAL
seuil_similarite_voie                                             0.85
nettoyage_flou                                                    True
sinistres                                                       100000
conformes_ban_avant                                              17516
conformes_ban_apres_nettoyage                                    17928
lignes_modifiees_par_nettoyage                                    5812
communes_completees                                               1902
regressions                                                         18
conformes_retenus                                                94362
en_qua

---
## Annexe · Retirer le lot `R14`

Supprime **uniquement** les objets créés par ce notebook (tables `_r014`, lignes du lot `R14` dans `dq.*`). Le lot de base n'est pas touché. À lancer seulement pour repartir de zéro.

In [60]:
SUPPRIMER_LOT = False          # passer à True pour retirer le lot R14, puis relancer cette cellule
if SUPPRIMER_LOT:
    assert LOT != BATCH
    for t in ["crm_cible.sinistre_r014", "gold.sinistre_certifie_r014", "silver.sinistre_r014", "silver.sinistre_r014_trace"]:
        executer(f"DROP TABLE IF EXISTS {t}")
    for t in ["dq.rejets", "dq.dq_history", "dq.certificat_qualite"]:
        executer(f"DELETE FROM {t} WHERE batch_id = :lot", lot=LOT)
    print(f"Lot {LOT} retiré : tables _r014 supprimées, lignes dq.rejets / dq.dq_history / dq.certificat_qualite supprimées.")
else:
    print("SUPPRIMER_LOT = False : rien n'est supprimé.")

SUPPRIMER_LOT = False : rien n'est supprimé.
